In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

data = np.array([
    [1480, 20.2, 52000],
    [1478, 20.3, 51800],
    [1425, 19.0, 48000]
])

normalized = scaler.fit_transform(data)

def cosine_similarity(a, b):
    return np.dot(a, b) / np.linalg.norm(a) * np.linalg.norm(b)

normal = normalized[0]
similar = normalized[1]
very_different = normalized[2]

print("Normal vs similar", cosine_similarity(normal, similar))
print("Normal vs very_different", cosine_similarity(normal, very_different))

In [1]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from sklearn.preprocessing import normalize

data = np.array([
    [1480, 20.2, 52000],
    [1478, 20.3, 51800],
    [1425, 19.0, 48000]
])

# Normalize properly
data_normalized = normalize(data)

# Calculate similarity
similarities = cosine_similarity([data_normalized[0]], data_normalized)
print("Similarity scores:", similarities)

Similarity scores: [[1.         1.         0.99999925]]


In [2]:
import pandas as pd
import numpy as np

#generate readings
def generate_readings():
    rng = np.random.default_rng(seed=42)
    num_rows = 250

    data = {
        "HMT": rng.integers(low=1440, high=1510, size=num_rows),
        "Slag_Al2O3": rng.uniform(low=19.5, high=21.5, size=num_rows),
        "W/V": rng.integers(low=48000, high=56000, size=num_rows)
    }

    df = pd.DataFrame(data)
    flag_rows = pd.DataFrame({
        "HMT": np.concatenate([
            rng.integers(low=1420, high=1440, size=25),  # too low
            rng.integers(low=1510, high=1530, size=25)   # too high
        ]),
        "Slag_Al2O3": rng.uniform(low=19.5, high=21.5, size=50),
        "W/V": rng.integers(low=48000, high=56000, size=50)
    })

    df = pd.concat([df, flag_rows], ignore_index=True)

    conditions = [
    ((df["HMT"] > 1510) | (df["HMT"] < 1440) | (df["Slag_Al2O3"] > 21.5) | (df["Slag_Al2O3"] < 19.5)),
    ((((df["HMT"] > 1440) & (df["HMT"] < 1450)) | ((df["HMT"] > 1500) & (df["HMT"] < 1510))) | (((df["Slag_Al2O3"] > 19.5) & (df["Slag_Al2O3"] < 19.8)) | ((df["Slag_Al2O3"] > 21.2) & (df["Slag_Al2O3"] < 21.5)))) 
    ]

    output = ["flag_issue", "hold"]

    df["Output"] = np.select(conditions, output, default="increase_WV")
    return df 

reading = generate_readings()

def reading_to_text(row):
    temp = str(row['HMT'])
    slag = str(row['Slag_Al2O3'])
    wind = str(row['W/V'])
    output = str(row['Output'])
    text = f"""Blast furnace reading: HMT is {temp}°C (normal range 1440-1510), 
                    Slag Al2O3 is {slag}% (normal range 19.5-21.5), 
                    W/V is {wind} Nm³/Hr. Status: {output}"""
    return text

for _,row in reading.iterrows():
    print(reading_to_text(row))

Blast furnace reading: HMT is 1446°C (normal range 1440-1510), 
                    Slag Al2O3 is 20.48798163848904% (normal range 19.5-21.5), 
                    W/V is 55690 Nm³/Hr. Status: hold
Blast furnace reading: HMT is 1494°C (normal range 1440-1510), 
                    Slag Al2O3 is 20.15972242466557% (normal range 19.5-21.5), 
                    W/V is 49182 Nm³/Hr. Status: increase_WV
Blast furnace reading: HMT is 1485°C (normal range 1440-1510), 
                    Slag Al2O3 is 19.789048377732094% (normal range 19.5-21.5), 
                    W/V is 48690 Nm³/Hr. Status: hold
Blast furnace reading: HMT is 1470°C (normal range 1440-1510), 
                    Slag Al2O3 is 19.706805935445104% (normal range 19.5-21.5), 
                    W/V is 55489 Nm³/Hr. Status: hold
Blast furnace reading: HMT is 1470°C (normal range 1440-1510), 
                    Slag Al2O3 is 20.675289144355425% (normal range 19.5-21.5), 
                    W/V is 52440 Nm³/Hr. Status: incre

In [4]:
from anthropic import Anthropic
from dotenv import load_dotenv
import os

load_dotenv()
client = Anthropic()

first_row = next(reading.iterrows())[1]
text = reading_to_text(first_row)

print(text)

Blast furnace reading: HMT is 1446°C (normal range 1440-1510), 
                    Slag Al2O3 is 20.48798163848904% (normal range 19.5-21.5), 
                    W/V is 55690 Nm³/Hr. Status: hold


In [5]:
import voyageai
import os
from dotenv import load_dotenv

load_dotenv()
api_Key = os.getenv("VOYAGEAI_API_KEY")

vo = voyageai.Client(api_key=api_Key)

response = vo.embed([text], model="voyage-3")

embedding = response.embeddings[0]
print(f"Embedding dimensions: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")

Embedding dimensions: 1024
First 5 values: [-0.02271309308707714, -0.00633853767067194, -0.008583436720073223, -0.04190477728843689, 0.04260905832052231]


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import anthropic
#building the RAG

#step 1
texts = []
for _, row in reading.iterrows():
    texts.append(reading_to_text(row))
    
vo = voyageai.Client(api_key=api_Key)

response = vo.embed(texts, model="voyage-3")
embeddings = response.embeddings

print(f"Total embeddings: {len(embeddings)}")
print(f"Each embedding dimensions: {len(embeddings[0])}")

#step 2: build the search function
def find_similar(query, embeddings, texts, top_n=10):
    #first change the query from text to embedding
    query_embedding = vo.embed([query], model="voyage-3").embeddings[0]

    #then compare the query embedding to the embeddings using cosine similarity
    similarities = cosine_similarity([query_embedding], embeddings)[0] #need to flatten as the result will be a 2D array

    #then i need the top n with the highest cosine similarities
    partitioned_indices = np.argpartition(similarities, -top_n)[-top_n:]
    top = partitioned_indices[np.argsort(similarities[partitioned_indices])[::-1]]

    print("Top recommendations:", top)
    return [texts[i] for i in top]

#find_similar("HMT is critically low and temperature dropped", embeddings, texts)

#step3: call claude
def ask_about_history(question):
    #first I need to find the similarities
    similar_responses = find_similar(question, embeddings, texts)

    #then I need call the client
    client = anthropic.Anthropic()

    #then create the message
    message = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            system="you are a blast furnace operator and you have been provided the historical context.",
            messages=[
                {"role": "user", "content": f"in the past we have seen the following {similar_responses}. provide a recommendation based on {question}"}
            ]
        )

    print(message.content[0].text)

ask_about_history("HMT is critically low and temperature dropped")



Total embeddings: 300
Each embedding dimensions: 1024
Top 3: [253 270 255 254 271 269 266 274 262 258]
# Blast Furnace Operational Recommendation

## Critical Issue Analysis

**Primary Problem: HMT Temperature Critically Low**

All 10 historical readings show **HMT consistently below the normal range (1440-1510°C)**, with readings ranging from **1421-1438°C**. This represents a **2-19°C deficit** from the minimum acceptable operating temperature.

---

## Immediate Actions Required

### 1. **Increase Blast Temperature & Volume**
- Gradually increase the hot blast temperature from the stove
- The W/V ratio is already variable (48,000-55,000 Nm³/Hr); consider increasing towards the upper range
- *Rationale:* Direct heat input will raise HMT most effectively

### 2. **Optimize Fuel Injection**
- Increase PCI (Pulverized Coal Injection) or coke breeze rates
- Verify fuel quality and combustion efficiency
- *Rationale:* More exothermic reactions = higher furnace temperature

### 3. **Check 